# Stress Prediction — 최종 후보 비교 (gender 제거 + quantile 재최적화 + 시드평균)

오늘 나온 "안전한 개선" 후보들을 조합해서 비교합니다.

- **A. v34 원본** (기준, gender 포함, quantile 0.54/0.48, weight 0.28, 시드 42 하나)
- **B. gender만 tree에서 제거** (오늘 가장 뚜렷했던 신호, LOO/누적제거 둘 다 확인됨)
- **C. B + quantile 재최적화** (grid search 1위 근처였던 0.50/0.52, weight 0.24)
- **D. B + 시드 2개 평균** (42, 1 — 분산 감소, 이론상 나빠질 이유 없음)

CV로 4개를 비교해서 가장 낮은 걸 최종 제출 파일로 만듭니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
N_ESTIMATORS = 1200
PAIR_QUANTILE_ORIG = 0.48
PAIR_WEIGHT_ORIG = 0.28
TREE_QUANTILE_ORIG = 0.54

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_predictions_matrix(train_features, val_features, target, seed) -> np.ndarray:
    """개별 트리 예측 행렬 (quantile과 무관, 한 번만 계산해서 재사용)."""
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=seed, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    return np.column_stack([t.predict(val_matrix) for t in model.estimators_])


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target, pair_quantile) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), pair_quantile, axis=1)


## 5-Fold CV — A/B/C/D 네 후보 동시 비교

tree는 (gender 포함 시드42), (gender 제외 시드42), (gender 제외 시드1) 세 가지만 계산하면
A/B/C/D를 전부 만들 수 있습니다 (quantile은 사후 계산이라 재학습 불필요).


In [3]:

train = pd.read_csv(DATA_DIR / "train.csv")
raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=42)

oof_A = np.zeros(len(y))  # 원본: gender 포함, 시드42, quantile 0.54/0.48, weight 0.28
oof_B = np.zeros(len(y))  # gender 제거, 시드42, quantile 0.54/0.48, weight 0.28
oof_C = np.zeros(len(y))  # gender 제거, 시드42, quantile 0.50/0.52, weight 0.24
oof_D = np.zeros(len(y))  # gender 제거, 시드42+1 평균, quantile 0.54/0.48, weight 0.28

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)

    train_with_gender = train_feat
    val_with_gender = val_feat
    train_no_gender = train_feat.drop(columns=["gender"])
    val_no_gender = val_feat.drop(columns=["gender"])

    tree_matrix_with_gender_s42 = tree_predictions_matrix(train_with_gender, val_with_gender, y_train, 42)
    tree_matrix_no_gender_s42 = tree_predictions_matrix(train_no_gender, val_no_gender, y_train, 42)
    tree_matrix_no_gender_s1 = tree_predictions_matrix(train_no_gender, val_no_gender, y_train, 1)

    pred_pair_orig_q = pair_prediction(train_feat, val_feat, y_train, PAIR_QUANTILE_ORIG)
    pred_pair_new_q = pair_prediction(train_feat, val_feat, y_train, 0.52)

    tree_A = np.quantile(tree_matrix_with_gender_s42, TREE_QUANTILE_ORIG, axis=1)
    tree_B = np.quantile(tree_matrix_no_gender_s42, TREE_QUANTILE_ORIG, axis=1)
    tree_C = np.quantile(tree_matrix_no_gender_s42, 0.50, axis=1)
    tree_D_s42 = tree_B
    tree_D_s1 = np.quantile(tree_matrix_no_gender_s1, TREE_QUANTILE_ORIG, axis=1)
    tree_D = np.mean([tree_D_s42, tree_D_s1], axis=0)

    oof_A[val_idx] = np.clip(np.round((1 - PAIR_WEIGHT_ORIG) * tree_A + PAIR_WEIGHT_ORIG * pred_pair_orig_q, 2), 0, 1)
    oof_B[val_idx] = np.clip(np.round((1 - PAIR_WEIGHT_ORIG) * tree_B + PAIR_WEIGHT_ORIG * pred_pair_orig_q, 2), 0, 1)
    oof_C[val_idx] = np.clip(np.round((1 - 0.24) * tree_C + 0.24 * pred_pair_new_q, 2), 0, 1)
    oof_D[val_idx] = np.clip(np.round((1 - PAIR_WEIGHT_ORIG) * tree_D + PAIR_WEIGHT_ORIG * pred_pair_orig_q, 2), 0, 1)

    print(f"Fold {fold} done")

summary = pd.DataFrame([
    {"candidate": "A. v34 원본", "cv_mae": mean_absolute_error(y, oof_A)},
    {"candidate": "B. gender 제거", "cv_mae": mean_absolute_error(y, oof_B)},
    {"candidate": "C. gender제거 + quantile재최적화", "cv_mae": mean_absolute_error(y, oof_C)},
    {"candidate": "D. gender제거 + 시드2개평균", "cv_mae": mean_absolute_error(y, oof_D)},
]).sort_values("cv_mae").reset_index(drop=True)

display(summary)


Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done


,candidate,cv_mae
0,B. gender 제거,0.147650
1,D. gender제거 + 시드2개평균,0.147773
2,C. gender제거 + quantile재최적화,0.148007
3,A. v34 원본,0.148033


## 1위 후보로 최종 제출 파일 생성

위 결과에서 `cv_mae`가 가장 낮은 후보의 알파벳을 `BEST_CANDIDATE`에 넣고 실행하세요.


In [ ]:

BEST_CANDIDATE = "B"  # "A", "B", "C", "D" 중 결과 1위로 수정

test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
raw_X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)

use_gender = BEST_CANDIDATE == "A"
train_final = raw_X if use_gender else raw_X.drop(columns=["gender"])
test_final = raw_X_test if use_gender else raw_X_test.drop(columns=["gender"])

seeds = [42, 1] if BEST_CANDIDATE == "D" else [42]
tree_matrices = [tree_predictions_matrix(train_final, test_final, y, s) for s in seeds]

tree_q = 0.50 if BEST_CANDIDATE == "C" else TREE_QUANTILE_ORIG
pair_q = 0.52 if BEST_CANDIDATE == "C" else PAIR_QUANTILE_ORIG
weight = 0.24 if BEST_CANDIDATE == "C" else PAIR_WEIGHT_ORIG

tree_preds = [np.quantile(m, tree_q, axis=1) for m in tree_matrices]
final_tree = np.mean(tree_preds, axis=0)
final_pair = pair_prediction(raw_X, raw_X_test, y, pair_q)
final_prediction = np.clip(np.round((1 - weight) * final_tree + weight * final_pair, 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / f"submit_final_candidate_{BEST_CANDIDATE}_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))
